In [1]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
!pip install -q openai langchain-openai langchain-community faiss-cpu python-dotenv gradio

In [3]:
# 환경 설정
import os
from dotenv import load_dotenv
import pandas as pd
import json
import time
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
# 예시가 성능의 70%를 결정한다는 연구결과도 있음.
# 예시
  # 다양성 (bias 되는걸 방지)
  # 관련성
  # 균형

### Fewshot
참고 논문:
- prompt를 어떤 순서로 넣느냐에 따른 성능 차이
  - 링크: https://arxiv.org/pdf/2104.08786
- example을 전달할 때, bias들이 발견되었다는 내용
  - 링크: https://arxiv.org/pdf/2102.09690
  - 긍정만 또는 부정만 많을 경우 좋지않음.
- 예시를 줄때, KNN(비슷한것끼리 묶는 알고리즘)을 이용해 묶어 프롬프트에 넣었더니 성능 증가
  - 링크: https://arxiv.org/pdf/2101.06804
- few shot example을 줄 때, 라벨을 random으로 줄때 성능 변화가 없었다? example로부터 배우는건 포맷에 치중된게 아닐까 하는 의문점 제시
  - 링크: https://arxiv.org/pdf/2202.12837
- example을 선택할 때 다양성 극대화 하는 방법 제안
  - 링크: https://arxiv.org/pdf/2209.01975

  

In [4]:
# 랭체인에서도 fewshot prompt template라는 클래스를 제공함.

from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate, ChatPromptTemplate, FewShotChatMessagePromptTemplate

1. PromptTemplate의 고유 인자
- input_variables: 프롬프트 템플릿 안에서 변수로 사용할 이름들의 리스트입니다. (랭체인이 이 리스트와 템플릿 내의 중괄호 {} 변수가 일치하는지 검사합니다.)
template: 실제 프롬프트의 형태를 정의하는 문자열입니다.
2. FewShotPromptTemplate의 고유 인자
- examples: 퓨샷(Few-shot) 학습에 사용할 예시 데이터 리스트입니다.
- example_prompt: 위 examples의 각 데이터를 어떤 형태의 문자열로 변환할지 지정하는 PromptTemplate 객체입니다.
- prefix: 예시들이 시작되기 전에 맨 처음에 올 지시문(Instruction)입니다.
- suffix: 예시들이 다 끝난 후, 맨 마지막에 올 문자열입니다. 주로 실제 사용자의 입력값을 받는 부분이 여기에 들어갑니다.
- input_variables: 최종적으로 이 퓨샷 프롬프트가 실행될 때 외부에서 받아야 할 변수명입니다. (위 코드에서는 마지막 suffix에 들어갈 ['input'] 하나만 필요합니다.)
- example_separator: 각 예시(example_prompt로 변환된 문자열들) 사이를 구분할 때 쓸 문자열입니다. 보통 줄바꿈(\n\n)을 씁니다.

In [6]:
examples = [
    {"input" : "이 제품 정말 최고입니다!", "output": "긍정"},
    {"input" : "이 제품 품질이 너무 안좋아요", "output": "부정"},
    {"input" : "보통이에요. 무난합니다.", "output": "중립"},
    {"input" : "디자인은 좋은데 성능이 아쉬워요", "output": "혼합"},
]

In [7]:
example_template = PromptTemplate(
    input_variables = ['input', 'output'],
    template = "리뷰: {input}\n감정: {output}"
)

fewshot_prompt = FewShotPromptTemplate(
    examples = examples,
    example_prompt = example_template,
    prefix = '다음 예시를 참고해서 리뷰의 감정을 분류해주세요\n',
    suffix = "\n리뷰: {input}\n감정:",
    input_variables = ['input'],
    example_separator = "\n\n"
)

In [8]:
formatted = fewshot_prompt.format(input = '가격은 비싸지만 품질이 뛰어나요')
print(formatted)

다음 예시를 참고해서 리뷰의 감정을 분류해주세요


리뷰: 이 제품 정말 최고입니다!
감정: 긍정

리뷰: 이 제품 품질이 너무 안좋아요
감정: 부정

리뷰: 보통이에요. 무난합니다.
감정: 중립

리뷰: 디자인은 좋은데 성능이 아쉬워요
감정: 혼합


리뷰: 가격은 비싸지만 품질이 뛰어나요
감정:


In [9]:
example_prompt_chat = ChatPromptTemplate.from_messages({
    ('human', '리뷰:{input}'),
    ('ai', '감정: {output}')
})

fewshot_chat_prompt = FewShotChatMessagePromptTemplate(
    examples = examples,
    example_prompt = example_prompt_chat

)

In [11]:
final_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 감정 분석 전문가입니다. 리뷰의 감정을 긍정/부정/중립/혼합 중 한가지로 판단해주세요.'),
    fewshot_chat_prompt,
    ('human', '리뷰ㅣ {input}')
])

In [15]:
chain = final_prompt | llm
chain.invoke({'input':'가격은 비싸지만 품질이 뛰어나요'}).content

'감정: 혼합'

In [16]:
large_examples = [
    {"input": "정말 만족합니다. 강력 추천!", "output": "긍정"},
    {"input": "배송이 빠르고 포장이 꼼꼼해요.", "output": "긍정"},
    {"input": "가격 대비 훌륭합니다.", "output": "긍정"},
    {"input": "품질이 최악입니다. 환불 요청했어요.", "output": "부정"},
    {"input": "고객센터 응대가 너무 불친절합니다.", "output": "부정"},
    {"input": "택배가 분실되었어요.", "output": "부정"},
    {"input": "보통이에요. 특별한 점은 없습니다.", "output": "중립"},
    {"input": "사진과 동일한 제품입니다.", "output": "중립"},
    {"input": "디자인은 예쁜데 내구성이 약해요.", "output": "혼합"},
    {"input": "기능은 많은데 사용법이 복잡합니다.", "output": "혼합"},
]

In [20]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_core.vectorstores import InMemoryVectorStore

In [22]:
from langchain_openai import OpenAIEmbeddings

example_selector = SemanticSimilarityExampleSelector.from_examples(
    large_examples,
    OpenAIEmbeddings(model="text-embedding-3-small"),
    InMemoryVectorStore,
    k=3
)

#### InMemoryVectorStore에 대한 설명
- 데이터를 메모리(RAM)에 임시로 저장하는 벡터 저장소
- 특징:
  - 휘발성: 세션이 종료되거나 런타임이 다시 시작되면 데이터가 사라집니다. (디스크에 저장하지 않음)
  - 빠른 속도: 로컬 메모리에서 작동하므로 매우 빠릅니다.
  - 간편함: 별도의 DB 설치나 서버 설정 없이 즉석에서 벡터 검색(유사도 검색) 기능을 구현할 때 사용합니다.
- 현재 코드에서는 large_examples 리스트에 있는 문장들을 OpenAIEmbeddings를 통해 숫자로 변환한 뒤, 이 저장소에 넣어두고 사용자의 질문과 가장 비슷한 예시를 'K-최근접 이웃(KNN)' 알고리즘으로 찾아내는 역할을 합니다.

In [25]:
queries = [
    '배송이 너무 느려요',
    '값은 좀 나가지만 성능은 훌륭해요',
    '무난한 제품이에요'

]

for q in queries:
  selected = example_selector.select_examples({'input': q})
  print(f"query: {q}")
  for s in selected:
    print(f"  selected : {s['output']}, {s['input']}")

query: 배송이 너무 느려요
  selected : 긍정, 배송이 빠르고 포장이 꼼꼼해요.
  selected : 부정, 고객센터 응대가 너무 불친절합니다.
  selected : 부정, 택배가 분실되었어요.
query: 값은 좀 나가지만 성능은 훌륭해요
  selected : 긍정, 가격 대비 훌륭합니다.
  selected : 혼합, 기능은 많은데 사용법이 복잡합니다.
  selected : 혼합, 디자인은 예쁜데 내구성이 약해요.
query: 무난한 제품이에요
  selected : 중립, 사진과 동일한 제품입니다.
  selected : 부정, 품질이 최악입니다. 환불 요청했어요.
  selected : 긍정, 가격 대비 훌륭합니다.


In [26]:
eval_dataset = [
    {"text": "완벽한 제품입니다! 강력 추천합니다.", "expected": "긍정"},
    {"text": "두 번 다시 구매하지 않겠습니다.", "expected": "부정"},
    {"text": "평범합니다. 특별히 좋지도 나쁘지도 않아요.", "expected": "중립"},
    {"text": "기능은 좋은데 AS가 아쉬워요.", "expected": "혼합"},
    {"text": "가격도 착하고 품질도 좋아요.", "expected": "긍정"},
    {"text": "사진과 너무 달라서 실망했습니다.", "expected": "부정"},
    {"text": "그냥 쓸만 합니다.", "expected": "중립"},
    {"text": "배송은 빠른데 제품이 기대 이하에요.", "expected": "혼합"},
]

In [27]:
def evaluate_zero_shot(dataset):
  results = []
  for item in dataset:
    response = llm.invoke([
        SystemMessage(content = "리뷰의 감정을 분류하세요. 반드시 '긍정','부정','중립','혼합' 중 하나만 출력하세요."),
        HumanMessage(content = item['text'])
    ]).content

    predicted = response.strip()
    results.append({
        'text' : item['text'],
        'expected' : item['expected'],
        'predicted' : predicted,
        'correct' : predicted == item['expected'],
    })

  return results

In [29]:
def evaluate_few_shot(dataset, examples):
    results = []
    base_messages = [SystemMessage(content = "리뷰의 감정을 분류하세요. 반드시 '긍정', '부정', '중립', '혼합' 중 하나만 출력하세요")]
    for ex in examples:
        base_messages.append(HumanMessage(content = ex['input']))
        base_messages.append(AIMessage(content = ex['output']))

    for item in dataset:
        messages = base_messages + [HumanMessage(content = item['text'])]
        response = llm.invoke(messages).content

        predicted = response.strip()
        results.append({
            'text' : item['text'],
            'expected' : item['expected'],
            'predicted' : predicted,
            'correct' : predicted == item['expected'],
        })

    return results

In [34]:
few_shot_examples = [
    {"input": "완벽한 제품입니다! 강력 추천합니다.", "output": "긍정"},
    {"input": "두 번 다시 구매하지 않겠습니다.", "output": "부정"},
    {"input": "평범합니다. 특별히 좋지도 나쁘지도 않아요.", "output": "중립"},
    {"input": "기능은 좋은데 AS가 아쉬워요.", "output": "혼합"},
]

In [28]:
import pandas as pd

In [36]:
fewshot_results = evaluate_few_shot(eval_dataset,few_shot_examples)

In [37]:
pd.DataFrame(fewshot_results)

,text,expected,predicted,correct
0,완벽한 제품입니다! 강력 추천합니다.,긍정,긍정,True
1,두 번 다시 구매하지 않겠습니다.,부정,부정,True
2,평범합니다. 특별히 좋지도 나쁘지도 않아요.,중립,중립,True
3,기능은 좋은데 AS가 아쉬워요.,혼합,혼합,True
4,가격도 착하고 품질도 좋아요.,긍정,긍정,True
5,사진과 너무 달라서 실망했습니다.,부정,부정,True
6,그냥 쓸만 합니다.,중립,중립,True
7,배송은 빠른데 제품이 기대 이하에요.,혼합,혼합,True


In [ ]:
# 프롬프트 또는 모델에 대한 정량적 평가를 해야함.
# 예시 정답지 등에 대해 점수를 매기던가 classification 등을 잘 수행했는지 등의 평가지표로 LLM 서비스의 성능을 높일 수 있음.

번역, 요약, 문구 생성 등은 BLUE 또는 LOUGE 스코어 등이 있음

말이 25마리, 한번에 5마리만 경주를 할 수 있다.
가장 빠른 3마리 말을 찾는게 목표,
몇번의 경주를 해야될까요?

In [38]:
result = llm.invoke('말이 25마리 있습니다. 한번에 5마리만 경주를 할 수 있습니다. 가장 빠른 3마리 말을 찾으려면 몇번의 경주를 해야할까요?')

In [39]:
print(result.content)

25마리의 말 중에서 가장 빠른 3마리를 찾기 위해서는 다음과 같이 경주를 진행할 수 있습니다.

1. 처음에 5마리씩 5번 경주를 시행하여 각 경주의 순위를 기록합니다. 이로써 총 5개의 그룹이 생깁니다. (경주 1~5)

2. 각 경주에서 1등 말을 기록합니다. 그러면 5마리의 1등 말을 모을 수 있습니다.

3. 이제 이 5마리의 1등 말을 가지고 또 한 경주를 진행합니다. 이 경주에서 1등 말을 기록하여 누가 가장 빠른지 확인합니다. (경주 6)

4. 경주 6의 결과에 따라 상위 3마리를 찾기 위해 추가 경주를 진행해야 합니다. 1등에서 3등까지의 말을 찾기 위해서는 1등 말을 제외한 경주 6에서 2등, 3등의 말을 오후로 각 경주에서 1등했던 5마리의 조별 2등과 3등 말을 고려해야 합니다.

5. 그러면 2등과 3등의 말을 찾기 위해 경주 6에서 다른 말들의 경주 결과를 통해 순위를 고려하여 추가적으로 1~2번 경주했던 말을 포함하여 진행할 수 있습니다.

대략적으로 이러한 방법으로 하여 추가 경주가 필요하게 됩니다. 

결국 이 과정을 반복하여 3마리를 찾기 위해 약 7번의 경주가 필요하게 될 것입니다. 

경주 1~5: 각 5마리씩 5번 
경주 6: 1등 비교 
경주 7: 추가적인 3위 후보와 상위 성적 비교 

그래서 총 7번의 경주가 필요합니다.


In [45]:
print(llm.invoke('3개 문 중에 하나의 황금이 있고, 나머지는 야채입니다. 당신이 1번 문을 고르고, 사회자가 2번 문으로 바꾸시겠습니까? 라고 묻습니다. 사회자는 아직 어떤 문도 열지 않았는데, 바꾸시겠습니까? 라고 묻습니다. 바꾸는게 유리할까요?').content)

네, 바꾸는 것이 유리합니다. 이 상황은 "몽티 홀 문제"라고 알려진 확률 퍼즐과 비슷합니다.

처음에 3개의 문 중 하나를 선택했을 때, 선택한 문 뒤에 황금이 있을 확률은 1/3이고, 나머지 2개의 문 중 하나에 황금이 있을 확률은 2/3입니다. 

사회자가 2번 문으로 바꾸겠냐고 묻는 상황에서, 사회자가 이미 어떤 문도 열지 않았기 때문에, 여전히 1번 문에 황금이 있을 확률은 1/3이고, 2번 문 또는 3번 문 중 하나에 황금이 있을 확률은 2/3입니다. 

즉, 1번 문을 고른 후 그 선택을 유지하면 황금을 얻을 확률은 1/3으로 남아있고, 바꾼다면 황금을 얻을 확률은 2/3로 증가합니다. 따라서 바꾸는 것이 더 유리합니다.


In [41]:
# 한번에 답이 나오는 문제가 아니라, 추론이 필요한 문제가 힘들수 있음.

### 프롬프트를 어떻게 원하는 대로 유도할 수 있는지
- CoT (Chain of Thought)
 - 처음 나온 논문: https://arxiv.org/pdf/2205.11916
 - "단계별로 생각해보세요" 라는 프롬프트만 추가했는데 성능 증가

- llm이 정답을 맞추기 난해한 문제들을 모은 사이트: https://www.llm-quiz.com/quiz

In [46]:
print(llm.invoke('3개 문 중에 하나의 황금이 있고, 나머지는 야채입니다. 당신이 1번 문을 고르고, 사회자가 2번 문으로 바꾸시겠습니까? 라고 묻습니다. 사회자는 아직 어떤 문도 열지 않았는데, 바꾸시겠습니까? 라고 묻습니다. 바꾸는게 유리할까요? 단계별로 생각해 봅시다.').content)

이 문제는 "몬티 홀 문제(Monty Hall problem)"라고 알려져 있는 확률 퍼즐입니다. 단계별로 생각해 보겠습니다.

1. **초기 선택**: 당신은 3개의 문 중 1번 문을 선택했습니다. 이때 각각의 문 뒤에 있는 내용은 다음과 같습니다.
   - 1번 문: 1/3의 확률로 황금이 있음
   - 2번 문: 1/3의 확률로 황금이 있음
   - 3번 문: 1/3의 확률로 황금이 있음

2. **사회자의 행동**: 사회자는 당신이 선택하지 않은 문 중에서 황금이 없는 문을 하나 열어야 합니다. 만약 당신이 1번 문을 선택했을 때, 사회자는 2번 문이나 3번 문 중 하나를 선택해야 하고, 그 문에는 무조건 야채가 있습니다.

3. **문을 바꿀지 결정하기**: 이제 당신은 1번 문에 남아있거나, 사회자가 열지 않은 다른 문으로 바꿀 수 있습니다. 상황을 분석해 보겠습니다.

   - **1번 문에 남는 경우**: 당신은 선택한 1번 문이 실제로 황금이 있을 확률은 여전히 1/3입니다.
   
   - **다른 문(2번 or 3번)으로 바꾸는 경우**: 사회자가 문을 열 때, 항상 황금이 없는 문이 열리므로, 선택 후 남은 문(2번이나 3번)을 고를 경우 황금이 있을 확률이 2/3입니다. 이는 1번 문에서 황금이 아닐 확률(2/3)이 그대로 전환되어 다른 문에 할당되기 때문입니다.

4. **결론**: 따라서, 문을 바꾸는 것이 더 유리합니다. 선택한 문(1번 문)에 남아 있는 것보다, 사회자가 열지 않은 다른 문으로 바꾸는 것이 황금을 얻을 확률이 높습니다. 바꿀 경우 황금이 있을 확률은 2/3, 남을 경우는 1/3이므로, 바꾸는 것이 더 유리합니다.


In [47]:
# CoT
# 단계별로 생각해보세요. 봅시다..
# 지문의 조건을 모두 나열 -> 각각의 조건들을 검토하면서 추론해줘/풀어줘
# 이 문제가 기존 유명한 문제와 어떻게 다른지를 분석하고 답해줘.

print(llm.invoke('3개 문 중에 하나의 황금이 있고, 나머지는 야채입니다. 당신이 1번 문을 고르고, 사회자가 2번 문으로 바꾸시겠습니까? 라고 묻습니다. 사회자는 아직 어떤 문도 열지 않았는데, 바꾸시겠습니까? 라고 묻습니다. 바꾸는게 유리할까요? 각각의 조건들을 검토하면서 추론해줘/풀어줘.').content)

이 문제는 고전적인 "몬티 홀 문제"와 비슷합니다. 대략적인 설정은 다음과 같습니다:

1. 세 개의 문이 있습니다: 문 A, 문 B, 문 C.
2. 그 중 하나의 문 뒤에는 황금이 있고, 나머지 두 개의 문 뒤에는 야채가 있습니다.
3. 당신은 문 A를 선택했습니다.
4. 사회자는 문 B를 선택적으로 보여줍니다. 하지만 당신은 문 B를 고르지 않았으므로, 문 B 뒤에는 반드시 야채가 있어야 한다는 고려하에 문 B는 열리지 않습니다.

이제 당신에게 문 A에 그대로 남을 것인지, 아니면 문 C로 바꿀 것인지를 물어보는 상황입니다.

### 조건 검토

1. **선택한 문 A에 황금이 있을 확률:** 
   - 처음 선택할 때, 황금이 문 A 뒤에 있을 확률은 1/3입니다.

2. **문 A에 야채가 있을 확률:**
   - 초기 선택에서 문 A에 황금이 없을 확률은 2/3입니다. 즉, 문 B 또는 문 C 중 하나에 황금이 있을 확률이 2/3입니다. 이 경우, 사회자가 문 B를 보여줄 때, 문 C에 황금이 남아있는 경우가 됩니다.

3. **사회자가 문 B를 열 수 있는 조건:**
   - 당신이 문 A를 선택했을 때, 사회자는 항상 야채가 있는 문 B를 열 수 있습니다.

### 결론
- 문 A에 황금이 있을 확률: 1/3
- 문 C에 황금이 있을 확률: 2/3

따라서, 문 A에서 문 C로 바꾸는 것이 훨씬 유리합니다. 문 C로 변경할 경우, 황금을 얻을 확률은 2/3로 증가합니다. 그래서 바꾸는 것이 유리합니다.


In [48]:
# few-shot cot : CoT, 예시

In [50]:
system_prompt = """당신은 퍼즐 전문가입니다. 아래 문제들은 유명한 문제의 **변형**입니다.
원래 문제의 답을 그대로 적용하지 마세요. 이문제의 조건을 정확히 읽고 답하세요.

문제: 친구 집까지 평균 시속 3마일로 걸어갔습니다. 왕복 전체 평균 속도를 시속 6마일로 만들려면, 돌아올 때 얼마나 빨리 달려야 할까요
예시: 셔츠 1장을 말리는데 4시간이 걸립니다. 셔츠 5장을 말리면 몇시간?
[흔한 실수] 4 x 5 - 20시간
[올바른 생각] 동시에 널면 됩니다. 답: 4시간

이처럼 문제의 조건을 주의 깊게 읽고 단계별로 생각하세요.
"""

In [49]:
"친구 집까지 평균 시속 3마일로 걸어갔습니다. 왕복 전체 평균 속도를 시속 6마일로 만들려면, 돌아올 때 얼마나 빨리 달려야 할까요"

'친구 집까지 평균 시속 3마일로 걸어갔습니다. 왕복 전체 평균 속도를 시속 6마일로 만들려면, 돌아올 때 얼마나 빨리 달려야 할까요'

In [52]:
print(llm.invoke(system_prompt).content)

이 문제를 해결하기 위해서는 왕복 거리와 시간을 고려해야 합니다.

1. **가정**: 친구 집까지의 거리 D(마일)라고 가정해 봅시다.
2. **첫 번째 구간**: 친구 집에 가는 데 걸린 시간은 \( D / 3 \) 시간입니다.
3. **전체 왕복 거리**: 왕복 거리는 \( 2D \)입니다. 
4. **원하는 전체 평균 속도**: 평균 속도가 6마일로 되어야 하므로 전체 시간을 T라고 하면,
   \[
   2D / T = 6
   \]
   따라서, 전체 시간을 구하면,
   \[
   T = 2D / 6 = D / 3
   \]

5. **시간 계산**: 친구 집에 가는 시간은 이미 \( D / 3 \)로 계산되었고, 돌아올 때의 시간은 \( T - (D / 3) \)입니다. 우리가 구하는 것은 돌아오는 데 걸리는 시간입니다. 두 개의 시간은 아래와 같습니다.

   - 친구 집에 가는 시간: \( D / 3 \)
   - 돌아오는 시간: \( T - (D / 3) = D / 3 - (D / 3) = 0 \) (이것은 불가능한 상황입니다)

이 부분에서 문제가 생기는 것을 알 수 있습니다. 즉, 시속 3마일로 걸어가고, 왕복 평균 속도를 6마일로 만들 수 없습니다. 

결론적으로, 친구 집으로 돌아올 때는 무한대의 속도로 달려야만 평균 속도가 시속 6마일이 될 수 있습니다. 현실적으로 불가능한 상황이므로, 문제는 유효한 해답이 없음을 암시합니다.
